# 第一阶段 步骤05：反向传播的理论知识

> 来源：《深度学习入门2：自制框架》（斋藤康毅 著，郑明智 译，人民邮电出版社 2023）
> 目标：从零构建深度学习框架 **DeZero** 的第五步。

---

## 核心目标

这一步是**纯理论**：讲清反向传播背后的数学依据——**链式法则**，以及"为什么沿输出→输入方向传播导数最高效"。

（本步不新增框架代码，`backward` 方法要等到步骤6 才落地。）

## 5.1 链式法则

链式法则用于计算**复合函数的导数**：复合函数的导数 = 外层函数的导数 × 内层函数的导数。

对 $y = C(B(A(x)))$，令 $a = A(x)$、$b = B(a)$，则：

$$\frac{dy}{dx} = \frac{dy}{db} \cdot \frac{db}{da} \cdot \frac{da}{dx}$$

链式法则是反向传播的理论基础，也是整个框架的**精髓**所在。

## 5.2 反向传播的推导

复合函数的导数可分解为**各函数导数的乘积**，但乘积的**先后顺序并没有规定**，可以自由选择。

本书选择**从输出到输入**的方向（与正向相反）：

1. 从输出端的导数（$=1$）开始；
2. 依次乘以沿途各函数的导数；
3. 最终得到 $\frac{dy}{dx}$。

传播的是 **"y 对各变量的导数"**（$\frac{\partial y}{\partial x}$、$\frac{\partial y}{\partial a}$、$\frac{\partial y}{\partial b}$…），即把 y 当作"主角"。

> 补充：若按输入→输出方向计算则叫**前向模式**（forward mode），在步骤10 的专栏里有介绍。

In [ ]:
import numpy as np

# 承接步骤04：Variable、Function、Square、Exp、numerical_diff
class Variable:
    def __init__(self, data):
        self.data = data

class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        return output
    def forward(self, x):
        raise NotImplementedError()

class Square(Function):
    def forward(self, x):
        return x ** 2

class Exp(Function):
    def forward(self, x):
        return np.exp(x)

def numerical_diff(f, x, eps=1e-4):
    x0 = Variable(x.data - eps)
    x1 = Variable(x.data + eps)
    y0 = f(x0)
    y1 = f(x1)
    return (y1.data - y0.data) / (2 * eps)

# 复合函数 y = (e^(x^2))^2 = C(B(A(x)))
def f(x):
    A = Square(); B = Exp(); C = Square()
    return C(B(A(x)))

x = Variable(np.array(0.5))

# ① 数值微分直接求 dy/dx
print("数值微分:", numerical_diff(f, x))   # 3.2974426293330694

# ② 链式法则：从输出端往回，逐层相乘
a = x.data ** 2          # A: a = x^2
b = np.exp(a)            # B: b = e^a

dy_db = 2 * b            # C: y = b^2 的导数 2b
db_da = np.exp(a)        # B: b = e^a 的导数 e^a
da_dx = 2 * x.data       # A: a = x^2 的导数 2x

dydx = dy_db * db_da * da_dx
print("链式法则:", dydx)   # 3.297442541400256（与数值微分一致）

## 5.3 用计算图表示

把求 $\frac{dy}{dx}$ 的流程画在计算图上，可以把"函数 × 它的导函数"合并成一个节点，从而简化反向传播图：

```
正向（左 → 右，算值）：   x ─[A]─▶ a ─[B]─▶ b ─[C]─▶ y
反向（右 → 左，算导数）： dy/dx ◀── dy/da ◀── dy/db ◀── 1
                         （每步乘以沿途函数的导函数）
```

- **正向传播**：沿计算图从左到右算值（$x \to a \to b \to y$）；
- **反向传播**：沿计算图从右到左传播"y 对各变量的导数"（从 $\frac{dy}{dy}=1$ 起步）。

可以理解为：变量既有**普通值**也有**导数值**；函数既有**普通计算**（正向）也有**求导计算**（反向）。

## 这一步的"为什么"

这一步没有改动任何代码，却回答了最核心的问题：**"为什么能高效地对每个变量求导"**。

答案是：**沿"输出→输入"方向、用链式法则逐层相乘，一次反向传播就能同时求出对所有变量的导数**。

这正是深度学习框架必须采用反向传播、而非数值微分的原因——数值微分对每个参数都要额外做两次前向计算，参数一多就不可接受。

---

> 预告：步骤6 开始**手动实现反向传播**——给 `Square`、`Exp` 各自加上 `backward` 方法，把这里的理论变成代码。